### **Loading Partitions & Defining Feature Groups**

In [118]:
import pandas as pd
import numpy as np

In [119]:
#  Load the pre-split, leakage-free partitions
train_df = pd.read_csv("../data/processed/train_encounters.csv")
test_df = pd.read_csv("../data/processed/test_encounters.csv")

print(f"Loaded Train encounters: {train_df.shape}")
print(f"Loaded Test encounters:  {test_df.shape}")

Loaded Train encounters: (79478, 50)
Loaded Test encounters:  (19862, 50)


In [120]:
#  Separate features (X) and binary target (y)
target_col = "target_readmit_30d"

# Non-predictive columns to exclude:
# - Identifiers (encounter_id, patient_nbr)
# - Original multiclass label (readmitted)
# - Target itself (target_readmit_30d)
cols_to_drop = ["encounter_id", "patient_nbr", "readmitted", target_col]

X_train = train_df.drop(columns=cols_to_drop)
y_train = train_df[target_col]

X_test = test_df.drop(columns=cols_to_drop)
y_test = test_df[target_col]

print(f"\nFeature matrix X_train shape: {X_train.shape}")
print(f"Feature matrix X_test shape:  {X_test.shape}")



Feature matrix X_train shape: (79478, 46)
Feature matrix X_test shape:  (19862, 46)


### **Isolating Numerical and Categorical Pipelines**

In [121]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [122]:
#  Explicitly identify numerical features
num_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

In [123]:
#  Identify categorical features (remaining object or integer code columns)
cat_cols = [c for c in X_train.columns if c not in num_cols]

print(f"Numerical features ({len(num_cols)}): {num_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols[:5]} ... (+{len(cat_cols)-5} more)")

Numerical features (8): ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
Categorical features (38): ['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id'] ... (+33 more)


In [124]:
#  Build sub-pipelines
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [125]:
#  Combine into a unified ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_cols),
        ("cat", cat_transformer, cat_cols)
    ],
    remainder="drop"
)

In [128]:
# Test fit-transform strictly on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\nSuccessfully fitted preprocessor!")
print(f"Transformed X_train shape: {X_train_processed.shape}")
print(f"Transformed X_test shape:  {X_test_processed.shape}")


Successfully fitted preprocessor!
Transformed X_train shape: (79478, 265)
Transformed X_test shape:  (19862, 265)


### **Establishing the Logistic Regression Baseline**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [129]:
#  Initialize Baseline Logistic Regression with balanced weighting
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
    solver="lbfgs"
)

In [ ]:
#  Train baseline model on preprocessed training data
log_reg.fit(X_train_processed, y_train)

#  Generate predictions and predicted probabilities on held-out test set
y_pred_lr = log_reg.predict(X_test_processed)
y_prob_lr = log_reg.predict_proba(X_test_processed)[:, 1]

In [130]:
#Calculate key metrics
lr_roc_auc = roc_auc_score(y_test, y_prob_lr)
lr_cm = confusion_matrix(y_test, y_pred_lr)

print("================ Logistic Regression Baseline Results ================")
print(f"ROC-AUC Score: {lr_roc_auc:.4f}")
print("\nConfusion Matrix [ [TN, FP], [FN, TP] ]:")
print(lr_cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=["No Early Readmit (0)", "Readmitted <30d (1)"]))

================ Logistic Regression Baseline Results ================
ROC-AUC Score: 0.6573

Confusion Matrix [ [TN, FP], [FN, TP] ]:
[[12048  5603]
 [ 1021  1190]]

Classification Report:
                      precision    recall  f1-score   support

No Early Readmit (0)       0.92      0.68      0.78     17651
 Readmitted <30d (1)       0.18      0.54      0.26      2211

            accuracy                           0.67     19862
           macro avg       0.55      0.61      0.52     19862
        weighted avg       0.84      0.67      0.73     19862



Training Candidate Models (Decision Tree & Random Forest)

In [131]:
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [132]:
# Decision Tree Comparison Baseline
# We constrain max_depth to prevent severe overfitting on the 273 features
dt_model = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",
    random_state=42
)
dt_model.fit(X_train_processed, y_train)

y_pred_dt = dt_model.predict(X_test_processed)
y_prob_dt = dt_model.predict_proba(X_test_processed)[:, 1]

dt_roc_auc = roc_auc_score(y_test, y_prob_dt)
print(f"Decision Tree (max_depth=6) ROC-AUC: {dt_roc_auc:.4f}")

Decision Tree (max_depth=6) ROC-AUC: 0.6535


In [139]:
# Primary Candidate Model: Random Forest Classifier
# n_estimators=150 provides sufficient ensemble stability
# max_depth=12 and min_samples_leaf=20 prevent individual trees from memorizing rare encounter noise
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth= 14,
    min_samples_leaf=20,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)

print("\nTraining Random Forest Classifier (this may take 20-40 seconds)...")
rf_model.fit(X_train_processed, y_train)

y_pred_rf = rf_model.predict(X_test_processed)
y_prob_rf = rf_model.predict_proba(X_test_processed)[:, 1]

rf_roc_auc = roc_auc_score(y_test, y_prob_rf)
print(f"Random Forest Classifier ROC-AUC:    {rf_roc_auc:.4f}")


Training Random Forest Classifier (this may take 20-40 seconds)...
Random Forest Classifier ROC-AUC:    0.6694


In [150]:
from sklearn.ensemble import HistGradientBoostingClassifier

# 1. Initialize HistGradientBoostingClassifier with balanced class weighting
gb_model = HistGradientBoostingClassifier(
    max_iter=100,
    max_depth=6,
    class_weight="balanced",
    random_state=42
)

# 2. Train on the baseline processed training data (clean 273 features)
print("Training Gradient Boosting model...")
gb_model.fit(X_train_processed, y_train)

# 3. Predict probabilities on our held-out test set
y_prob_gb = gb_model.predict_proba(X_test_processed)[:, 1]
gb_roc_auc = roc_auc_score(y_test, y_prob_gb)

print("================ Comparative Model Check ================")
print(f"Logistic Regression Baseline: {lr_roc_auc:.4f}")
print(f"Random Forest (Primary):      {rf_roc_auc:.4f}")
print(f"Gradient Boosting Classifier: {gb_roc_auc:.4f}")

Training Gradient Boosting model...
================ Comparative Model Check ================
Logistic Regression Baseline: 0.6573
Random Forest (Primary):      0.6694
Gradient Boosting Classifier: 0.6679


In [151]:
from sklearn.metrics import recall_score, precision_score, f1_score

# Construct unified comparison table
models_summary = pd.DataFrame({
    "Model": ["Logistic Regression (Baseline)", "Decision Tree (max_depth=5)", "Random Forest (Primary)"],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_dt),
        roc_auc_score(y_test, y_prob_rf)
    ],
    "Recall (Class 1)": [
        recall_score(y_test, y_pred_lr, pos_label=1),
        recall_score(y_test, y_pred_dt, pos_label=1),
        recall_score(y_test, y_pred_rf, pos_label=1)
    ],
    "Precision (Class 1)": [
        precision_score(y_test, y_pred_lr, pos_label=1),
        precision_score(y_test, y_pred_dt, pos_label=1),
        precision_score(y_test, y_pred_rf, pos_label=1)
    ],
    "F1-Score (Class 1)": [
        f1_score(y_test, y_pred_lr, pos_label=1),
        f1_score(y_test, y_pred_dt, pos_label=1),
        f1_score(y_test, y_pred_rf, pos_label=1)
    ]
})

print("================ Comprehensive Model Benchmark Table ================")
print(models_summary.to_string(index=False))

print("\nRandom Forest Confusion Matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nRandom Forest Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=["No Early Readmit (0)", "Readmitted <30d (1)"]))

================ Comprehensive Model Benchmark Table ================
                         Model  ROC-AUC  Recall (Class 1)  Precision (Class 1)  F1-Score (Class 1)
Logistic Regression (Baseline) 0.657306          0.538218             0.175180            0.264327
   Decision Tree (max_depth=5) 0.653487          0.608322             0.167039            0.262107
       Random Forest (Primary) 0.669404          0.545455             0.186832            0.278329

Random Forest Confusion Matrix [ [TN, FP], [FN, TP] ]:
[[12402  5249]
 [ 1005  1206]]

Random Forest Classification Report:
                      precision    recall  f1-score   support

No Early Readmit (0)       0.93      0.70      0.80     17651
 Readmitted <30d (1)       0.19      0.55      0.28      2211

            accuracy                           0.69     19862
           macro avg       0.56      0.62      0.54     19862
        weighted avg       0.84      0.69      0.74     19862



In [152]:
#  Retrieve all one-hot feature names generated by ColumnTransformer
feature_names = preprocessor.get_feature_names_out()

#  Extract feature importances from the fitted Random Forest
importances = rf_model.feature_importances_

feat_imp_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("Top 15 Most Influential Features in Random Forest:")
print(feat_imp_df.head(15).to_string(index=False))

Top 15 Most Influential Features in Random Forest:
                         Feature  Importance
           num__number_inpatient    0.196185
 cat__discharge_disposition_id_1    0.063990
            num__num_medications    0.050843
         num__num_lab_procedures    0.043434
           num__number_emergency    0.040278
           num__time_in_hospital    0.038754
           num__number_diagnoses    0.038320
cat__discharge_disposition_id_22    0.034385
             num__num_procedures    0.020679
          num__number_outpatient    0.014961
 cat__discharge_disposition_id_3    0.014245
   cat__payer_code_Missing_Payer    0.009660
                 cat__insulin_No    0.009539
                cat__age_[50-60)    0.008837
      cat__admission_source_id_7    0.008779


In [153]:
# Threshold optimization scan for Random Forest
thresholds = np.arange(0.35, 0.55, 0.02)
scan_results = []

for t in thresholds:
    y_pred_t = (y_prob_rf >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm_t.ravel()
    
    rec = recall_score(y_test, y_pred_t, pos_label=1)
    prec = precision_score(y_test, y_pred_t, pos_label=1, zero_division=0)
    f1 = f1_score(y_test, y_pred_t, pos_label=1)
    
    scan_results.append({
        "Threshold": round(t, 2),
        "Recall": round(rec, 4),
        "Precision": round(prec, 4),
        "F1-Score": round(f1, 4),
        "False Negatives": fn,
        "False Positives": fp,
        "True Positives": tp
    })

threshold_df = pd.DataFrame(scan_results)
print("================ Random Forest Threshold Optimization Scan ================")
print(threshold_df.to_string(index=False))

================ Random Forest Threshold Optimization Scan ================
 Threshold  Recall  Precision  F1-Score  False Negatives  False Positives  True Positives
      0.35  0.9593     0.1204    0.2140               90            15494            2121
      0.37  0.9294     0.1252    0.2207              156            14353            2055
      0.39  0.8874     0.1303    0.2272              249            13095            1962
      0.41  0.8476     0.1382    0.2377              337            11686            1874
      0.43  0.7987     0.1471    0.2484              445            10243            1766
      0.45  0.7345     0.1560    0.2573              587             8787            1624
      0.47  0.6649     0.1664    0.2661              741             7366            1470
      0.49  0.5834     0.1785    0.2734              921             5935            1290
      0.51  0.4930     0.1931    0.2775             1121             4555            1090
      0.53  0.4039     0

In [161]:
import os
import joblib
from pathlib import Path
from sklearn.pipeline import Pipeline

# 1. Package the fitted ColumnTransformer and our tuned Random Forest (max_depth=14)
final_readmission_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", rf_model)
])

# 2. Resolve the destination path robustly (ensuring app/ exists at project root)
# If running from notebooks/, resolve up to project root:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
app_dir = project_root / "app"
app_dir.mkdir(parents=True, exist_ok=True)
model_path = app_dir / "readmission_rf_pipeline.joblib"

# 3. Serialize artifact
joblib.dump(final_readmission_pipeline, model_path)
file_size_mb = os.path.getsize(model_path) / (1024 * 1024)

print(f"Artifact serialized to: {model_path.resolve()}")
print(f"File size: {file_size_mb:.2f} MB")

# 4. Smoke-test: Load back and predict on raw X_test encounters (verifying zero transformation leakage)
loaded_pipeline = joblib.load(model_path)
sample_raw_encounters = X_test.iloc[:5]

# Predict continuous risk probabilities
sample_probs = loaded_pipeline.predict_proba(sample_raw_encounters)[:, 1]

# Apply calibrated operational threshold (tau = 0.48)
OPERATIONAL_THRESHOLD = 0.48
sample_risk_flags = (sample_probs >= OPERATIONAL_THRESHOLD).astype(int)

print("\n--- Smoke Test Verification on Raw Data ---")
for i, (prob, flag) in enumerate(zip(sample_probs, sample_risk_flags)):
    status = "HIGH RISK (<30d)" if flag == 1 else "STANDARD RISK"
    print(f"Encounter {i+1}: P(Readmit) = {prob:.4f} -> {status}")

Artifact serialized to: C:\Users\HP\diabetic-readmission-prediction\app\readmission_rf_pipeline.joblib
File size: 10.44 MB

--- Smoke Test Verification on Raw Data ---
Encounter 1: P(Readmit) = 0.4708 -> STANDARD RISK
Encounter 2: P(Readmit) = 0.4740 -> STANDARD RISK
Encounter 3: P(Readmit) = 0.5125 -> HIGH RISK (<30d)
Encounter 4: P(Readmit) = 0.3555 -> STANDARD RISK
Encounter 5: P(Readmit) = 0.4516 -> STANDARD RISK


In [160]:
# 1. Operationalize calibrated decision boundary (tau = 0.49)
OPERATIONAL_THRESHOLD = 0.48

y_pred_calibrated = (y_prob_rf >= OPERATIONAL_THRESHOLD).astype(int)

print(f"================ Random Forest Operational Evaluation (Threshold = {OPERATIONAL_THRESHOLD}) ================")
print(f"Final ROC-AUC: {rf_roc_auc:.4f}")
print("\nConfusion Matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, y_pred_calibrated))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_calibrated, target_names=["No Early Readmit (0)", "Readmitted <30d (1)"]))

================ Random Forest Operational Evaluation (Threshold = 0.48) ================
Final ROC-AUC: 0.6694

Confusion Matrix [ [TN, FP], [FN, TP] ]:
[[10974  6677]
 [  825  1386]]

Classification Report:
                      precision    recall  f1-score   support

No Early Readmit (0)       0.93      0.62      0.75     17651
 Readmitted <30d (1)       0.17      0.63      0.27      2211

            accuracy                           0.62     19862
           macro avg       0.55      0.62      0.51     19862
        weighted avg       0.85      0.62      0.69     19862



In [ ]:
# 1. Create audit dataframe combining demographic metadata with actuals and predictions
subgroup_df = test_df[["race", "gender", "age"]].copy()
subgroup_df["y_true"] = y_test.values
subgroup_df["y_prob"] = y_prob_rf
subgroup_df["y_pred"] = (y_prob_rf >= OPERATIONAL_THRESHOLD).astype(int)

# 2. Audit Recall and False Negative Rate across Race subgroups with N >= 100
print("================ Demographic Subgroup Performance Audit (Race) ================")
race_audit = []
for group, grp_data in subgroup_df.groupby("race"):
    total_positives = grp_data["y_true"].sum()
    if total_positives >= 30:  # Minimum clinical sample size threshold
        rec = recall_score(grp_data["y_true"], grp_data["y_pred"], pos_label=1, zero_division=0)
        prec = precision_score(grp_data["y_true"], grp_data["y_pred"], pos_label=1, zero_division=0)
        race_audit.append({
            "Subgroup": group,
            "Total Encounters": len(grp_data),
            "Actual Readmissions": total_positives,
            "Subgroup Recall": round(rec, 4),
            "Subgroup Precision": round(prec, 4)
        })

print(pd.DataFrame(race_audit).to_string(index=False))

================ Demographic Subgroup Performance Audit (Race) ================
       Subgroup  Total Encounters  Actual Readmissions  Subgroup Recall  Subgroup Precision
AfricanAmerican              3695                  413           0.6126              0.1707
      Caucasian             14832                 1665           0.6414              0.1777
       Hispanic               465                   39           0.5128              0.1361
          Other               291                   34           0.4706              0.1860
   Unknown_Race               466                   47           0.2766              0.1444
